In [1]:
# Repro: Download steals shared CPU pool, starving Iterate|Extract
#
# Pipeline: list_files (instant) -> Download (ActorPool) -> Iterate (ActorPool) -> write
#
# Budget: 16 CPUs, reservation_ratio=0.5, 2 eligible ops
#   reserved/op = 4 CPUs  |  shared pool = 8 CPUs
#   Download ramps to 12 actors before producing any output -> exhausts shared pool
#   Iterate starts with 0 input and can never exceed its reserved 4 CPUs
#
# Bandwidth saturation (Praateek's "worst part"):
#   Node egress is fixed. With N download actors, each gets 1/N bandwidth.
#   12 actors x 10s = 1 actor x 0.83s = same 1.2 files/sec throughput.
#   Extra actors provide zero throughput gain but steal Iterate's shared budget.

import time, threading, collections
import ray, ray.data
from ray.data import ActorPoolStrategy

print(f"Ray version: {ray.__version__}")


Ray version: 2.56.0


In [2]:
NUM_FILES = 32

# Bandwidth saturation model: simulate peak (12 actors sharing node egress)
# 1 actor alone:  DOWNLOAD_DELAY_SOLO = 0.83s => 1.2 files/sec
# 12 actors peak: each takes 12 x 0.83 = 10s  => still 1.2 files/sec (same!)
DOWNLOAD_DELAY_SOLO  = 10.0 / 12   # ~0.83s with full bandwidth
DOWNLOAD_PEAK        = 12           # actors that exhaust shared pool (reserved=4 + shared=8)
DOWNLOAD_DELAY       = DOWNLOAD_DELAY_SOLO * DOWNLOAD_PEAK  # 10s (saturated case)

ITERATE_DELAY = 5.0  # per-item work; 4 reserved actors => 0.8 files/sec (vs 2.4 with 12)

print(f"Download: {DOWNLOAD_PEAK} actors x {DOWNLOAD_DELAY:.0f}s = {DOWNLOAD_PEAK/DOWNLOAD_DELAY:.1f} files/sec")
print(f"          1 actor  x {DOWNLOAD_DELAY_SOLO:.2f}s = {1/DOWNLOAD_DELAY_SOLO:.1f} files/sec  (same throughput!)")
print(f"Iterate starved  (4 actors): {4/ITERATE_DELAY:.1f} files/sec")
print(f"Iterate potential(12 actors): {12/ITERATE_DELAY:.1f} files/sec  ({12//4}x loss from starvation)")


Download: 12 actors x 10s = 1.2 files/sec
          1 actor  x 0.83s = 1.2 files/sec  (same throughput!)
Iterate starved  (4 actors): 0.8 files/sec
Iterate potential(12 actors): 2.4 files/sec  (3x loss from starvation)


In [3]:
import ray as _ray

@_ray.remote(num_cpus=0)
class _Counter:
    def __init__(self):
        self._alloc  = collections.defaultdict(int)
        self._active = collections.defaultdict(int)
        self._peak_alloc  = collections.defaultdict(int)
        self._peak_active = collections.defaultdict(int)
    def alloc(self, s):
        self._alloc[s] += 1
        self._peak_alloc[s] = max(self._peak_alloc[s], self._alloc[s])
    def dealloc(self, s):
        self._alloc[s] = max(0, self._alloc[s] - 1)
    def enter(self, s):
        self._active[s] += 1
        self._peak_active[s] = max(self._peak_active[s], self._active[s])
    def exit(self, s):
        self._active[s] = max(0, self._active[s] - 1)
    def snapshot(self):
        return {
            "alloc":        dict(self._alloc),
            "active":       dict(self._active),
            "peak_alloc":   dict(self._peak_alloc),
            "peak_active":  dict(self._peak_active),
        }

def list_files(batch):
    return {"file_id": list(range(NUM_FILES))}

def write(batch):
    return batch

class DownloadActor:
    def __init__(self):
        _ray.get_actor("_ctr").alloc.remote("dl")
    def __del__(self):
        try: _ray.get_actor("_ctr").dealloc.remote("dl")
        except Exception: pass
    def __call__(self, batch):
        ctr = _ray.get_actor("_ctr")
        ctr.enter.remote("dl")
        try:
            time.sleep(DOWNLOAD_DELAY)
            return batch
        finally:
            ctr.exit.remote("dl")

class IterateExtractActor:
    def __init__(self):
        _ray.get_actor("_ctr").alloc.remote("it")
    def __del__(self):
        try: _ray.get_actor("_ctr").dealloc.remote("it")
        except Exception: pass
    def __call__(self, batch):
        ctr = _ray.get_actor("_ctr")
        ctr.enter.remote("it")
        try:
            time.sleep(ITERATE_DELAY)
            return batch
        finally:
            ctr.exit.remote("it")


In [4]:
ray.shutdown()
ray.init(num_cpus=16)

ctr = _Counter.options(name="_ctr").remote()
snapshots = []
_stop = threading.Event()

def _monitor():
    t0 = time.perf_counter()
    total = ray.cluster_resources().get("CPU", 16)
    while not _stop.is_set():
        snap = ray.get(ctr.snapshot.remote())
        snapshots.append((
            time.perf_counter() - t0,
            snap["alloc"],
            snap["active"],
            total - ray.available_resources().get("CPU", total),
        ))
        _stop.wait(1.0)

t0 = time.perf_counter()
ds = (
    ray.data.from_items([{"seed": 0}])
    .map_batches(list_files, batch_size=1)
    .repartition(NUM_FILES)
    .map_batches(DownloadActor,       batch_size=1, num_cpus=1,
                 compute=ActorPoolStrategy(min_size=1, max_size=NUM_FILES))
    .map_batches(IterateExtractActor, batch_size=1, num_cpus=1,
                 compute=ActorPoolStrategy(min_size=1, max_size=NUM_FILES))
    .map_batches(write, batch_size=1, num_cpus=0)
)

monitor = threading.Thread(target=_monitor, daemon=True)
monitor.start()
results = ds.take_all()
_stop.set()
monitor.join(timeout=3)

peaks = ray.get(ctr.snapshot.remote())
print(f"Done: {len(results)} items in {time.perf_counter()-t0:.1f}s")
print(f"Peak actors — download: {peaks['peak_alloc'].get('dl',0)}  iterate: {peaks['peak_alloc'].get('it',0)}")


2026-07-14 18:42:31,156	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8269 


2026-07-14 18:42:33,493	INFO streaming_executor.py:193 -- Starting execution of Dataset dataset_5_0. Full logs are in /tmp/ray/session_2026-07-14_18-42-15_415223_1786482/logs/ray-data


2026-07-14 18:42:33,494	INFO streaming_executor.py:194 -- Execution plan of Dataset dataset_5_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(list_files)] -> AllToAllOperator[Repartition] -> ActorPoolMapOperator[MapBatches(DownloadActor)] -> ActorPoolMapOperator[MapBatches(IterateExtractActor)] -> TaskPoolMapOperator[MapBatches(write)]


[2026-07-14 18:42:33,529 E 1786482 1786482] core_worker.cc:2149: Actor with class name: 'MapWorker(MapBatches(DownloadActor))' and ID: '35bc8bf4f5733602567622b601000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.
2026-07-14 18:42:33,555	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.


2026-07-14 18:42:33,590	WARNING resource_manager.py:766 -- Cluster resources are not enough to run any task from TaskPoolMapOperator[MapBatches(list_files)]. The job may hang forever unless the cluster scales up.


...

2026-07-14 18:43:24,006	INFO logging_progress.py:231 -- MapBatches(write): 31/32


2026-07-14 18:43:24,006	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 8.0B object store


2026-07-14 18:43:24,007	INFO logging_progress.py:192 -- ============================================


2026-07-14 18:43:24,765	INFO streaming_executor.py:327 -- ✔️  Dataset dataset_5_0 execution finished in 51.27 seconds


Done: 32 items in 52.1s
Peak actors — download: 12  iterate: 10


In [5]:
# Budget constants (16 CPUs, ratio=0.5, 2 ops)
RESERVED = 4
SHARED   = 8
STARVED_AT = RESERVED + SHARED  # download actors needed to exhaust shared pool = 12

print("=== Timeline (1s samples) ===")
print(f"  {'t':>5}  {'dl_alloc':>8} {'dl_active':>9}  {'it_alloc':>8} {'it_active':>9}  {'cpu':>5}  note")
print("  " + "-"*85)

it_started = False
for ts, alloc, active, cpu in snapshots:
    dl = alloc.get("dl", 0)
    it = alloc.get("it", 0)
    dl_t = active.get("dl", 0)
    it_t = active.get("it", 0)

    note = ""
    if it == 0 and dl > 0:
        note = "<- iterate: 0 input"
    elif not it_started and it_t > 0:
        it_started = True
        stolen = max(0, dl - RESERVED)
        note = f"<- iterate FIRST task (dl={dl}, shared stolen={stolen})"
    elif dl >= STARVED_AT and it <= RESERVED:
        note = f"<- STARVED: iterate capped at reserved={RESERVED}"

    print(f"  {ts:5.1f}s  {dl:>8d} {dl_t:>9d}  {it:>8d} {it_t:>9d}  {cpu:>4.0f}  {note}")

print()

# Summary
peak_dl = max((s[1].get("dl", 0) for s in snapshots), default=0)
it_at_starvation = [s[1].get("it", 0) for s in snapshots if s[1].get("dl", 0) >= STARVED_AT]
peak_it_starved  = max(it_at_starvation, default=0)

print(f"Download peak: {peak_dl} actors  (exhausts shared when >= {STARVED_AT})")
print(f"Iterate during starvation: {peak_it_starved} actors (reserved={RESERVED}, could use {STARVED_AT} without stealing)")
print(f"Throughput loss: {STARVED_AT/ITERATE_DELAY:.1f} vs {peak_it_starved/ITERATE_DELAY:.1f} files/sec"
      f" => {STARVED_AT/max(1,peak_it_starved):.0f}x slower")
print()
print(f"Bandwidth saturation: {peak_dl} download actors x {DOWNLOAD_DELAY:.0f}s"
      f" = {peak_dl/DOWNLOAD_DELAY:.1f} files/sec"
      f" (same as 1 actor x {DOWNLOAD_DELAY_SOLO:.2f}s = {1/DOWNLOAD_DELAY_SOLO:.1f} files/sec)")
print(f"=> {peak_dl} actors wasted for zero throughput gain, starving Iterate")


=== Timeline (1s samples) ===
      t  dl_alloc dl_active  it_alloc it_active    cpu  note
  -------------------------------------------------------------------------------------
    0.2s         0         0         0         0     2  
    1.2s         1         0         1         0    11  
    2.2s         1         1         1         0     3  
    3.2s         2         2         1         0     4  
    4.2s         3         3         1         0     5  
    5.2s         5         5         1         0     7  
    6.2s         5         5         1         0     7  
    7.2s         5         5         1         0     7  
    8.2s         6         6         1         0     8  
    9.2s         7         6         1         0     8  
   10.3s         7         7         1         0    10  
   11.3s         7         7         1         0    10  
   12.3s         9         9         1         1    12  <- iterate FIRST task (dl=9, shared stolen=5)
   13.3s         9         9       